# CRUW 레이더 학습 준비 (VoD 지연 시 대안)

이 노트북은 **CRUW (Camera Radar With You)** 공개 데이터셋으로 레이더 기반 인지 모델을 실험하기 위한 **참고 문서 목록**, **데이터 구조 이해**, **경량 PyTorch 학습 루프 예시**를 담습니다. 본격적인 SOTA 재현은 공식 **RODNet** 저장소의 학습 스크립트를 사용하는 것이 좋습니다.

---

## CRUW 관련 공식·논문·코드 (서류/리소스)

| 구분 | 링크 | 비고 |
|------|------|------|
| 데이터셋 홈 | [https://www.cruwdataset.org/](https://www.cruwdataset.org/) | 소개, 다운로드 안내 |
| Introduction | [https://www.cruwdataset.org/introduction](https://www.cruwdataset.org/introduction) | 센서·시나리오·어노테이션 개요 |
| Resources (Devkit, RODNet) | [https://www.cruwdataset.org/resources](https://www.cruwdataset.org/resources) | **cruw-devkit**, **RODNet** GitHub |
| **cruw-devkit** | [https://github.com/yizhou-wang/cruw-devkit](https://github.com/yizhou-wang/cruw-devkit) | 캘리브레이션, RAMap↔좌표 변환, 시각화, ROD2021 튜토리얼 노트북 |
| **RODNet** (학습/추론 코드) | [https://github.com/yizhou-wang/RODNet](https://github.com/yizhou-wang/RODNet) | Range–Azimuth 히트맵 입력, 교차 모달 학습 |
| RODNet WACV 2021 | [https://openaccess.thecvf.com/content/WACV2021/html/Wang_RODNet_Radar_Object_Detection_Using_Cross-Modal_Supervision_WACV_2021_paper.html](https://openaccess.thecvf.com/content/WACV2021/html/Wang_RODNet_Radar_Object_Detection_Using_Cross-Modal_Supervision_WACV_2021_paper.html) | 회의 논문 |
| RODNet IEEE J-STSP (저널) | [https://ieeexplore.ieee.org/document/9353210](https://ieeexplore.ieee.org/document/9353210) / [arXiv:2102.05150](https://arxiv.org/abs/2102.05150) | 확장판 |
| CRUW·데이터셋 논문 (CVPRW 2021) | [WAD paper (OpenAccess)](https://openaccess.thecvf.com/content/CVPR2021W/WAD/html/Wang_Rethinking_of_Radars_Role_A_Camera-Radar_Dataset_and_Systematic_Annotator_CVPRW_2021_paper.html) | 데이터셋·어노테이터 체계 |
| ROD2021 Challenge | [CodaLab evaluation](https://codalab.lisn.upsaclay.fr/competitions/1063) | 벤치마크 제출(2022 재개 공지 있음) |
| Devkit Colab 튜토리얼 | [cruw_devkit_tutorial_rod2021.ipynb](https://colab.research.google.com/github/yizhou-wang/cruw-devkit/blob/master/tutorials/cruw_devkit_tutorial_rod2021.ipynb) | ROD2021 형식 읽기·평가 |

**라이선스:** CRUW는 [cruw-devkit README](https://github.com/yizhou-wang/cruw-devkit)에 명시된 대로 학술·비영리 연구용 제한이 있습니다. 사용 전 사이트 약관을 확인하세요.

---

## 원하시는 출력과 CRUW/레이더에서의 대응

| 목표 | CRUW에서의 일반적 대응 |
|------|------------------------|
| 표적 감지 | RAMap 상 객체 존재/중심 (RODNet 검출 또는 히트맵 회귀) |
| 표적까지 거리 | **range(m)** 또는 range 축 인덱스 → 거리 그리드와 캘리브레이션으로 환산 |
| 좌측/정면/우측 | **azimuth(rad)** 또는 azimuth 축 인덱스 (차량 전방 기준 각도 구간으로 구분 가능) |
| 접근 중 / 이탈 중 | 원시 chirp 스택에서 도플러(속도) 정보 추출·학습 — RODNet의 multi-chirp 경로 또는 **range–doppler** 표현 추가 모델링이 필요할 수 있음 |
| 이동 경로 | 시계열 RAMap 또는 검출 (range, azimuth) 시퀀스로 **궤적** 근사 (별도 트래킹/연속 프레임 손실) |

CRUW 메타데이터의 `radar_h` / `radar_v`는 `range`×`azimuth` 크기와 `n_chirps` 등이 JSON에 기재되어 있습니다 ([devkit README Annotation Format](https://github.com/yizhou-wang/cruw-devkit)).


## 환경 설정 (선택)

```bash
# 예: 별도 conda 환경
conda create -n cruw python=3.10 -y
conda activate cruw
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124  # GPU에 맞게 조정
pip install numpy matplotlib tqdm

# 데이터 도구 (로컬 클론 후)
# git clone https://github.com/yizhou-wang/cruw-devkit.git
# cd cruw-devkit && pip install -e .
```

아래 `CRUW_ROOT`에 압축 해제한 CRUW 데이터 루트 경로를 지정하세요.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Optional

import numpy as np

# ========= 사용자 설정 =========
# Windows 예시: r"D:\datasets\CRUW" — 실제 경로로 바꾸세요.
CRUW_ROOT = Path(os.environ.get("CRUW_ROOT", ""))  # 비어 있으면 더미 모드
SEQUENCE_JSON = CRUW_ROOT / "example" / "metadata.json"  # 실제 시퀀스의 json 경로로 교체

print("CRUW_ROOT:", CRUW_ROOT if CRUW_ROOT else "(미설정 — 더미 학습만 실행)")

### 어노테이션 로드 예시 (General CRUW JSON)

각 프레임의 `radar_h` / `radar_v`에 `centers`: `[range(m), azimuth(rad)]`, `center_ids`: `[range_idx, azimuth_idx]` 형식이 옵니다.

In [ ]:
def load_sequence_metadata(json_path: Path) -> Optional[dict]:
    if not json_path.is_file():
        return None
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)


meta = load_sequence_metadata(SEQUENCE_JSON)
if meta:
    print("dataset:", meta.get("dataset"), "seq:", meta.get("seq_name"), "n_frames:", meta.get("n_frames"))
    frame0 = meta["metadata"][0]
    rh = frame0.get("radar_h", {})
    print("radar_h keys:", rh.keys())
    oi = rh.get("obj_info", {})
    print("n_objects:", rh.get("n_objects"), "centers (range_m, az_rad):", oi.get("centers"))
else:
    print("JSON 없음 — 경로를 설정한 뒤 다시 실행하세요.")

### RAMap `.npy` 로드 (경로가 있을 때)

실제 파일명은 시퀀스별 `folder_name` / `frame_name`을 따릅니다.

In [ ]:
def load_radar_npy(meta: Optional[dict], frame_index: int = 0, polar: str = "h") -> Optional[np.ndarray]:
    if not meta or frame_index >= len(meta["metadata"]):
        return None
    fr = meta["metadata"][frame_index]
    key = "radar_h" if polar == "h" else "radar_v"
    rd = fr.get(key, {})
    folder = rd.get("folder_name")
    fname = rd.get("frame_name")
    if not folder or not fname:
        return None
    # 시퀀스 루트는 json과 같은 상위에 있다고 가정 (구조에 맞게 수정)
    p = SEQUENCE_JSON.parent / folder / fname
    if not p.is_file():
        return None
    return np.load(p)


arr = load_radar_npy(meta, 0, "h") if meta else None
if arr is not None:
    print("RAMap shape:", arr.shape, arr.dtype)
else:
    print("RAMap 없음 또는 경로 불일치")

---

## 경량 학습 예시: RAMap → 객체 히트맵 (PyTorch)

실제 RODNet은 여러 chirp·시퀀스와 복잡한 디코더를 사용합니다. 여기서는 **동일한 아이디어**(공간 히트맵 학습)를 최소 구현으로 보여, 데이터 연결 후 손쉽게 확장할 수 있게 합니다.

- 입력: `[B, C, H_range, W_azimuth]` (chirp를 채널로 쌓거나 평균)
- 출력: 각 객체 클래스별 2D confidence 맵 (여기서는 단일 채널 가우시안 타깃)

**본격 학습:** [RODNet 저장소](https://github.com/yizhou-wang/RODNet)의 `train.py` 및 설정 파일을 사용하세요.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


def gaussian_heatmap(h: int, w: int, cy: float, cx: float, sigma: float = 2.0) -> np.ndarray:
    yy, xx = np.ogrid[:h, :w]
    g = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * sigma**2))
    return g.astype(np.float32)


class DummyRAMapDataset(Dataset):
    """CRUW 미다운로드 시: 가짜 RAMap + 가짜 중심으로 학습 루프 검증."""

    def __init__(self, n: int = 256, h: int = 128, w: int = 128, n_chirp: int = 4):
        self.n = n
        self.h, self.w = h, w
        self.n_chirp = n_chirp

    def __len__(self) -> int:
        return self.n

    def __getitem__(self, idx: int):
        rng = np.random.default_rng(idx)
        x = rng.normal(size=(self.n_chirp, self.h, self.w)).astype(np.float32)
        cy = float(rng.uniform(8, self.h - 8))
        cx = float(rng.uniform(8, self.w - 8))
        target = gaussian_heatmap(self.h, self.w, cy, cx, sigma=3.0)
        return torch.from_numpy(x), torch.from_numpy(target[None, ...])  # [C,H,W], [1,H,W]


class TinyRadarHeatmapNet(nn.Module):
    def __init__(self, in_ch: int = 4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def train_one_epoch(model, loader, opt, criterion):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        opt.step()
        total += loss.item() * xb.size(0)
    return total / len(loader.dataset)


BATCH = 8
EPOCHS = 3
ds = DummyRAMapDataset(n=128, h=128, w=128, n_chirp=4)
loader = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=0)
model = TinyRadarHeatmapNet(in_ch=4).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for ep in range(EPOCHS):
    loss = train_one_epoch(model, loader, opt, criterion)
    print(f"epoch {ep+1}/{EPOCHS}  mse_loss={loss:.6f}")

print("더미 학습 완료. CRUW RAMap을 `Dataset`에 연결하면 동일 루프로 치환 가능합니다.")

### 접근/이탈·경로 표시를 모델에 넣을 때 (설계 메모)

1. **도플러(접근/이탈):** CRUW의 `.npy`가 chirp 차원을 포함하면, 시간축 FFT 또는 모델 내 **clutter map 대비 차분**으로 속도 부호를 학습할 수 있습니다. VoD의 `v_r`와 유사한 역할입니다.
2. **경로:** 연속 프레임에서 검출 (range_idx, az_idx) 또는 (range_m, az_rad)를 추출한 뒤 **Kalman 필터** 또는 **LSTM/Transformer**로 보정된 궤적을 그리면 시각화에 적합합니다.
3. **RODNet 연동:** 학습은 저장소 클론 후 `python`이 아닌 해당 repo의 스크립트가 표준이므로, 이 노트북은 **데이터 탐색 + 프로토타입**용으로 두고, 재현은 RODNet 쪽을 권장합니다.

```bash
# 예시 (RODNet 저장소 구조는 버전마다 다를 수 있음 — README 확인)
# git clone https://github.com/yizhou-wang/RODNet.git && cd RODNet
# pip install -r requirements.txt
# python train.py --config ...
```